# Readmission-DL — End-to-End Pipeline

This notebook trains a model to predict 30-day readmission and saves artifacts for inference.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score


## Preprocessing

In [ ]:
def fit_preprocessor(df):
    df = df.copy()

    df['admission_date'] = pd.to_datetime(df['admission_date'], errors='coerce')
    df['admission_month'] = df['admission_date'].dt.month
    df['admission_month'] = df['admission_month'].fillna(df['admission_month'].median())

    df['glucose_missing'] = df['glucose_level_mgdl'].isnull().astype(int)
    glucose_median = df['glucose_level_mgdl'].median()
    df['glucose_level_mgdl'] = df['glucose_level_mgdl'].fillna(glucose_median)

    df = df.fillna(0)

    df = df.drop(columns=['patient_id', 'admission_date'])

    y = df['readmitted_30d']
    X = df.drop(columns=['readmitted_30d'])

    X = pd.get_dummies(X)
    columns = X.columns

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled, y.values, scaler, columns, glucose_median


def transform(df, scaler, columns, glucose_median):
    df = df.copy()

    df['admission_date'] = pd.to_datetime(df['admission_date'], errors='coerce')
    df['admission_month'] = df['admission_date'].dt.month
    df['admission_month'] = df['admission_month'].fillna(df['admission_month'].median())

    df['glucose_missing'] = df['glucose_level_mgdl'].isnull().astype(int)
    df['glucose_level_mgdl'] = df['glucose_level_mgdl'].fillna(glucose_median)

    patient_ids = df['patient_id']

    df = df.fillna(0)

    df = df.drop(columns=['patient_id', 'admission_date'])

    X = pd.get_dummies(df)
    X = X.reindex(columns=columns, fill_value=0)

    return scaler.transform(X), patient_ids


## Model

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


## Training

In [ ]:
train_df = pd.read_csv('../data/train.csv')

X, y, scaler, columns, glucose_median = fit_preprocessor(train_df)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_val = torch.tensor(y_val, dtype=torch.float32).view(-1,1)

pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
pos_weight = torch.tensor([pos_weight.item()])

model = MLP(X_train.shape[1])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_auc = 0

for epoch in range(30):
    model.train()
    logits = model(X_train)
    loss = criterion(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X_val)).numpy().flatten()
        auc = roc_auc_score(y_val, probs)

    print(f"Epoch {epoch} | Loss {loss.item():.4f} | AUC {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), '../model.pt')


## Threshold Tuning

In [ ]:
model.load_state_dict(torch.load('../model.pt'))
model.eval()

with torch.no_grad():
    probs = torch.sigmoid(model(X_val)).numpy().flatten()

best_f1 = 0
best_t = 0

for t in np.arange(0.1, 0.9, 0.01):
    preds = (probs > t).astype(int)
    f1 = f1_score(y_val, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print('Best threshold:', best_t)


## Save Artifacts

In [ ]:
with open('../scaler.pkl','wb') as f:
    pickle.dump(scaler,f)

with open('../columns.pkl','wb') as f:
    pickle.dump(columns,f)

with open('../glucose_median.pkl','wb') as f:
    pickle.dump(glucose_median,f)

with open('../threshold.pkl','wb') as f:
    pickle.dump(best_t,f)
